In [1]:
# import wandb
import torch
import pandas as pd
import numpy as np
import zipfile
import requests
from pathlib import Path

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TORCH_USE_CUDA_DSA'] = '1'

# ---------------------------------------------------------------------------
# Box download URLs — fill these in with your shared links
# Box direct download format: https://company.box.com/shared/static/<id>
# ---------------------------------------------------------------------------
BOX_URLS = {
    "os1": "https://virginia.box.com/shared/static/9tz0fm7kq6745qzgy22hia985uf6c3hz.zip",   # e.g. "https://company.box.com/shared/static/abc123"
    "os2": "https://virginia.box.com/shared/static/t5dt2l6dgc3oz8oeizj83e8938gx69yr.zip",
    "fs":  "https://virginia.box.com/shared/static/u1lmcljm3o75gseghv5acnd4417qrkn2.zip",
}

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path("..").resolve()
print(PROJECT_ROOT)
DATA_DIR     = PROJECT_ROOT / "data" / "classification"
print(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)


def download_and_extract(key: str, out_dir: Path) -> None:
    """Download a zip from Box and extract it to out_dir."""
    url = BOX_URLS[key]
    if not url:
        print(f"[{key}] No URL provided, skipping.")
        return
    if out_dir.exists() and any(out_dir.iterdir()):
        print(f"[{key}] Already extracted, skipping.")
        return
    zip_path = DATA_DIR / f"{key}.zip"
    print(f"[{key}] Downloading...")
    dl_url = url + ("?dl=1" if "?" not in url else "&dl=1")
    with requests.get(dl_url, stream=True) as r:
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    print(f"[{key}] Extracting...")
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(out_dir)
    zip_path.unlink()
    print(f"[{key}] Done -> {out_dir}")


# ---------------------------------------------------------------------------
# Download image folders
# ---------------------------------------------------------------------------
download_and_extract("os1", DATA_DIR / "os1")
download_and_extract("os2", DATA_DIR / "os2")
download_and_extract("fs",  DATA_DIR / "fs")

# ---------------------------------------------------------------------------
# Load CSVs (already in repo at data/classification/)
# ---------------------------------------------------------------------------
opensar1 = pd.read_csv(DATA_DIR / "opensar1_labels.csv")
opensar2 = pd.read_csv(DATA_DIR / "opensar2_labels.csv")
fusar    = pd.read_csv(DATA_DIR / "fusar_labels.csv")

# Resolve local image paths
opensar1["path"] = opensar1["path"].apply(lambda p: str(PROJECT_ROOT / p))
opensar2["path"] = opensar2["path"].apply(lambda p: str(PROJECT_ROOT / p))
fusar["path"]    = fusar["path"].apply(lambda p: str(PROJECT_ROOT / p))

print(f"OpenSARShip 1: {len(opensar1)} rows")
print(f"OpenSARShip 2: {len(opensar2)} rows")
print(f"FuSARShip:     {len(fusar)} rows")

fusar.head()

/Users/arocha/Downloads/2025/MSDS/Spring/ds6050_final_project-1
/Users/arocha/Downloads/2025/MSDS/Spring/ds6050_final_project-1/data/classification
[os1] Already extracted, skipping.
[os2] Already extracted, skipping.
[fs] Already extracted, skipping.
OpenSARShip 1: 11346 rows
OpenSARShip 2: 18864 rows
FuSARShip:     5101 rows


,fs_id,path,class_label,ship_type,polarization,label,is_cargo,google_path
0,fs_0001,/Users/arocha/Downloads/2025/MSDS/Spring/ds605...,Cargo/AggregatesCarrier,NaN,NaN,cargo,1,/content/drive/MyDrive/classification/fs/fs_00...
1,fs_0002,/Users/arocha/Downloads/2025/MSDS/Spring/ds605...,Cargo/AggregatesCarrier,NaN,NaN,cargo,1,/content/drive/MyDrive/classification/fs/fs_00...
2,fs_0003,/Users/arocha/Downloads/2025/MSDS/Spring/ds605...,Cargo/BulkCarrier,NaN,NaN,cargo,1,/content/drive/MyDrive/classification/fs/fs_00...
3,fs_0004,/Users/arocha/Downloads/2025/MSDS/Spring/ds605...,Cargo/BulkCarrier,NaN,NaN,cargo,1,/content/drive/MyDrive/classification/fs/fs_00...
4,fs_0005,/Users/arocha/Downloads/2025/MSDS/Spring/ds605...,Cargo/BulkCarrier,NaN,NaN,cargo,1,/content/drive/MyDrive/classification/fs/fs_00...


In [2]:
opensar1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11346 entries, 0 to 11345
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   os1_id        11346 non-null  object
 1   path          11346 non-null  object
 2   class_label   11346 non-null  object
 3   ship_type     11346 non-null  int64 
 4   polarization  11346 non-null  object
 5   label         11346 non-null  object
 6   is_cargo      11346 non-null  int64 
 7   google_path   11346 non-null  object
dtypes: int64(2), object(6)
memory usage: 709.3+ KB


In [3]:
opensar1['label'].value_counts()

label
cargo          8240
tanker         1670
unknown         744
other           384
engineering     270
passenger        38
Name: count, dtype: int64

In [4]:
8240/11346*100

72.62471355543803

In [5]:
from sklearn.model_selection import train_test_split
from torchvision import transforms


def make_leak_safe_splits_and_oversample_non_cargo(
    df: pd.DataFrame,
    label_col: str = "label",
    cargo_label: str = "cargo",
    path_col: str = "path",
    val_size: float = 0.15,
    test_size: float = 0.15,
    non_cargo_target_multiplier: float = 1.0,
    random_state: int = 42,
):
    """
    1) Split FIRST (train/val/test) to avoid leakage.
    2) Oversample non-cargo on TRAIN ONLY.
    3) Tag oversampled rows so you can apply stronger train-only augmentation.
    """
    if label_col not in df.columns:
        raise ValueError(f"Column '{label_col}' not found.")
    if path_col not in df.columns:
        raise ValueError(f"Column '{path_col}' not found.")

    work_df = df.copy().reset_index(drop=True)

    # Keep one row per file path so identical samples cannot leak across splits.
    work_df = work_df.drop_duplicates(subset=[path_col]).reset_index(drop=True)

    trainval_df, test_df = train_test_split(
        work_df,
        test_size=test_size,
        stratify=work_df[label_col],
        random_state=random_state,
    )

    val_ratio_within_trainval = val_size / (1.0 - test_size)
    train_df, val_df = train_test_split(
        trainval_df,
        test_size=val_ratio_within_trainval,
        stratify=trainval_df[label_col],
        random_state=random_state,
    )

    train_df = train_df.copy().reset_index(drop=True)
    val_df = val_df.copy().reset_index(drop=True)
    test_df = test_df.copy().reset_index(drop=True)

    train_df["is_oversampled"] = 0
    val_df["is_oversampled"] = 0
    test_df["is_oversampled"] = 0

    cargo_train = train_df[train_df[label_col] == cargo_label]
    non_cargo_train = train_df[train_df[label_col] != cargo_label]

    if len(cargo_train) == 0 or len(non_cargo_train) == 0:
        raise ValueError("Train split must contain both cargo and non-cargo rows.")

    target_non_cargo_size = int(round(len(cargo_train) * non_cargo_target_multiplier))

    if target_non_cargo_size > len(non_cargo_train):
        extra_needed = target_non_cargo_size - len(non_cargo_train)
        extra = non_cargo_train.sample(n=extra_needed, replace=True, random_state=random_state).copy()
        extra["is_oversampled"] = 1
        train_balanced = pd.concat([train_df, extra], axis=0, ignore_index=True)
    else:
        train_balanced = train_df

    train_balanced = train_balanced.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

    return train_balanced, val_df, test_df


# Train-only augmentation (stronger) vs clean val/test transforms
RESNET_MEAN = [0.485, 0.456, 0.406]
RESNET_STD = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15, fill=0),
    transforms.ToTensor(),
    transforms.Normalize(mean=RESNET_MEAN, std=RESNET_STD),
])

transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=RESNET_MEAN, std=RESNET_STD),
])


# Example usage
train_df, val_df, test_df = make_leak_safe_splits_and_oversample_non_cargo(
    opensar1,
    label_col="label",
    cargo_label="cargo",
    path_col="path",
    val_size=0.15,
    test_size=0.15,
    non_cargo_target_multiplier=1.0,
    random_state=42,
)

print("Train (after oversampling):")
print(train_df["label"].value_counts())
print("\nValidation:")
print(val_df["label"].value_counts())
print("\nTest:")
print(test_df["label"].value_counts())
print("\nOversampled rows in train:", int(train_df["is_oversampled"].sum()))

Train (after oversampling):
label
cargo          5768
tanker         3141
unknown        1380
other           697
engineering     482
passenger        68
Name: count, dtype: int64

Validation:
label
cargo          1236
tanker          251
unknown         111
other            57
engineering      41
passenger         6
Name: count, dtype: int64

Test:
label
cargo          1236
tanker          250
unknown         112
other            58
engineering      40
passenger         6
Name: count, dtype: int64

Oversampled rows in train: 3594


In [7]:
train_df, val_df, test_df = make_leak_safe_splits_and_oversample_non_cargo(
    opensar2,
    label_col="label",
    cargo_label="cargo",
    path_col="path",
    val_size=0.15,
    test_size=0.15,
    non_cargo_target_multiplier=1.0,
    random_state=42,
)

print("Train (after oversampling):")
print(train_df["label"].value_counts())
print("\nValidation:")
print(val_df["label"].value_counts())
print("\nTest:")
print(test_df["label"].value_counts())
print("\nOversampled rows in train:", int(train_df["is_oversampled"].sum()))

Train (after oversampling):
label
cargo          7625
tanker         3722
unknown        2456
other           860
engineering     458
passenger       129
Name: count, dtype: int64

Validation:
label
cargo          1634
tanker          586
unknown         388
other           131
engineering      70
passenger        21
Name: count, dtype: int64

Test:
label
cargo          1634
tanker          586
unknown         388
other           131
engineering      70
passenger        21
Name: count, dtype: int64

Oversampled rows in train: 2046


In [6]:
train_df, val_df, test_df = make_leak_safe_splits_and_oversample_non_cargo(
    fusar,
    label_col="label",
    cargo_label="cargo",
    path_col="path",
    val_size=0.15,
    test_size=0.15,
    non_cargo_target_multiplier=1.0,
    random_state=42,
)

print("Train (after oversampling):")
print(train_df["label"].value_counts())
print("\nValidation:")
print(val_df["label"].value_counts())
print("\nTest:")
print(test_df["label"].value_counts())
print("\nOversampled rows in train:", int(train_df["is_oversampled"].sum()))

Train (after oversampling):
label
other          1775
cargo          1480
tanker          174
engineering      97
passenger        44
Name: count, dtype: int64

Validation:
label
other          380
cargo          317
tanker          37
engineering     21
passenger       10
Name: count, dtype: int64

Test:
label
other          381
cargo          317
tanker          37
engineering     21
passenger       10
Name: count, dtype: int64

Oversampled rows in train: 0


In [8]:
fusar['label'].value_counts()

label
other          2536
cargo          2114
tanker          248
engineering     139
passenger        64
Name: count, dtype: int64

In [9]:
fusar.loc[fusar['label'] == 'other']

,fs_id,path,class_label,ship_type,polarization,label,is_cargo,google_path
2178,fs_2179,/Users/arocha/Downloads/2025/MSDS/Spring/ds605...,Fishing/BulkCarrier,NaN,NaN,other,0,/content/drive/MyDrive/classification/fs/fs_21...
2179,fs_2180,/Users/arocha/Downloads/2025/MSDS/Spring/ds605...,Fishing/Fishing,NaN,NaN,other,0,/content/drive/MyDrive/classification/fs/fs_21...
2180,fs_2181,/Users/arocha/Downloads/2025/MSDS/Spring/ds605...,Fishing/Fishing,NaN,NaN,other,0,/content/drive/MyDrive/classification/fs/fs_21...
2181,fs_2182,/Users/arocha/Downloads/2025/MSDS/Spring/ds605...,Fishing/Fishing,NaN,NaN,other,0,/content/drive/MyDrive/classification/fs/fs_21...
2182,fs_2183,/Users/arocha/Downloads/2025/MSDS/Spring/ds605...,Fishing/Fishing,NaN,NaN,other,0,/content/drive/MyDrive/classification/fs/fs_21...
...,...,...,...,...,...,...,...,...
5096,fs_5097,/Users/arocha/Downloads/2025/MSDS/Spring/ds605...,WingInGrnd/WingInGrnd,NaN,NaN,other,0,/content/drive/MyDrive/classification/fs/fs_50...
5097,fs_5098,/Users/arocha/Downloads/2025/MSDS/Spring/ds605...,WingInGrnd/WingInGrnd,NaN,NaN,other,0,/content/drive/MyDrive/classification/fs/fs_50...
5098,fs_5099,/Users/arocha/Downloads/2025/MSDS/Spring/ds605...,WingInGrnd/WingInGrnd,NaN,NaN,other,0,/content/drive/MyDrive/classification/fs/fs_50...
5099,fs_5100,/Users/arocha/Downloads/2025/MSDS/Spring/ds605...,WingInGrnd/WingInGrnd,NaN,NaN,other,0,/content/drive/MyDrive/classification/fs/fs_51...
